# 01 — Coleta e Curadoria de Dados

**CPZÃO · Auditoria de Calçadas · 3 classes**

Este notebook obtém as imagens rotuladas a partir de três bases públicas, aplica a curadoria
(deduplicação, conversão máscara→caixa, unificação de taxonomia) e monta o dataset YOLO com
split `train/val/test`.

| Classe | Fonte | Anotação original |
|:--|:--|:--|
| `buraco_trinca` | HF `Ryukijano/Pothole-detection-Yolov8` | caixas YOLO |
| `piso_tatil` | GitHub `qweawq/Blind-road-and-crosswalk-dataset` | máscara por pixel |
| `obstaculo` | HF `Abtinzandi/Obstacle-Detection-Dataset-YOLO` | caixas YOLO (12 classes → 1) |

> No Colab, faça upload da pasta do projeto ou clone o repositório antes de rodar.

In [ ]:
# Ambiente (Colab) — pule esta célula se estiver rodando localmente
import os, sys
if 'google.colab' in sys.modules:
    # instala o minimo pra rodar a coleta no Colab
    !pip -q install ultralytics huggingface_hub opencv-python-headless
    # opções: subir o zip do projeto pelo painel lateral ou clonar o repositório
    # !git clone <URL_DO_SEU_REPOSITORIO>.git projeto
    # os.chdir('projeto')
# garante que estamos na raiz do projeto (senao os caminhos relativos quebram)
assert os.path.exists('requirements.txt'), 'Rode a partir da raiz do repositório'
!pip -q install -r requirements.txt

In [ ]:
# 1) baixa as 3 bases públicas (cache local em _scratch/raw)
%run src/download_sources.py

In [ ]:
# 2) curadoria + split + dataset.yaml (o grosso do trabalho mora aqui)
%run src/build_dataset.py

In [ ]:
# 3) auditoria de integridade das imagens do dataset final (pega JPEG quebrado)
%run src/check_images.py data/dataset/images/*/*.jpg

### Estatísticas do dataset

In [ ]:
# le o resumo gerado pelo build_dataset e mostra numeros + tabela por classe
import json, pandas as pd
s = json.load(open('docs/dataset_stats.json'))['stats']
print('total de imagens :', s['total_imagens'])
print('total instâncias :', s['total_instancias'])
print('por split        :', s['splits'])
pd.DataFrame(s['classes']).T

### Amostra visual com as caixas (QA)

In [ ]:
# mostra os mosaicos de conferencia (imagens com as caixas desenhadas) de cada split
from IPython.display import Image, display
for split in ('train', 'val', 'test'):
    display(Image(filename=f'docs/figures/qa_{split}.jpg', width=900))

### Decisões de curadoria
* **máscara → caixa**: fechamento morfológico + componentes conectados (área mín. 0,15%);
* **unificação**: 12 classes de mobiliário urbano do ROD → `obstaculo`;
* **deduplicação** por *average hash* (Hamming ≤ 8): frames quase idênticos ficam no **mesmo** split;
* **split estratificado** 70/20/10 respeitando os grupos de duplicatas;
* **integridade**: JPEG truncado é descartado na coleta.